# NLP x Spatial Cross-Theme Analysis: Did Corporate Language Predict Expansion?

*Connecting 30 years of 10-K filings with Manhattan's store network*

---

This notebook bridges **Theme 1 (NLP corpus analysis)** and **Theme 2 (Manhattan spatial analysis)** by asking a deceptively simple question: **does the language Starbucks uses in its SEC filings predict where and how it expands?**

We trace two parallel timelines — one built from topic-modeled 10-K filings (1996-2025), the other from store counts and Manhattan's physical footprint — and test whether corporate language leads, lags, or merely mirrors strategic action.

**Preview of the punchline:** *The 10-K is more mirror than crystal ball — corporate language tends to describe strategy already in motion rather than telegraphing what comes next.*

### Series context

This is the capstone cross-theme notebook in the Starbucks analytics series:

- **Notebook 0** — [Manhattan Cafe Wars](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors): spatial EDA of Starbucks vs 1,200+ competitors
- **Notebook 1** — [10-K NLP Topic Modeling](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis): LDA topics and keyword trends across 30 years of filings
- **Notebook 1F** — [LDA Topic Explorer](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-lda-topic-explorer-pyldavis): interactive pyLDAvis visualization
- **Notebook 2A** — [Spatial Clustering](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering): DBSCAN and LISA cluster analysis
- **Notebook 2B** — [Location Fitness](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness): scoring each store's spatial fitness
- **Notebook 2C** — [Walk-Distance Analysis](https://www.kaggle.com/code/shiratoriseto/starbucks-walk-distance-analysis-osmnx): OSMnx network-based walk distances
- **Notebook C** — [Data Pipeline](https://www.kaggle.com/code/shiratoriseto/starbucks-data-pipeline-edgar-osm-to-csv): EDGAR + OSM data extraction
- **Notebook D** — [US Expansion Choropleth](https://www.kaggle.com/code/shiratoriseto/starbucks-us-expansion-animated-choropleth): animated state-level expansion
- **Notebook G** — **NLP x Spatial (this notebook)**

## Section 0 — Setup & Data Loading

We load two datasets — the NLP corpus (topic proportions, keyword time series, store counts from 10-K filings) and the Manhattan spatial dataset (enriched store locations, OSM cafe inventory). Together they span 30 fiscal years (1996-2025) and 171 Manhattan Starbucks locations.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from scipy import stats
from IPython.display import display, HTML

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Data path resolution ────────────────────────────────────────────
ON_KAGGLE = Path("/kaggle/working").exists()

if ON_KAGGLE:
    _candidates_nlp = [
        Path("/kaggle/input/starbucks-30year-nlp-corpus"),
        Path("/kaggle/input/datasets/shiratoriseto/starbucks-30year-nlp-corpus"),
    ]
    NLP_DIR = None
    for _p in _candidates_nlp:
        if _p.exists():
            NLP_DIR = _p
            break
    if NLP_DIR is None:
        import os
        avail = os.listdir("/kaggle/input") if Path("/kaggle/input").exists() else []
        raise FileNotFoundError(f"NLP dataset not found. Available: {avail}")

    _candidates_spatial = [
        Path("/kaggle/input/manhattan-cafe-wars"),
        Path("/kaggle/input/datasets/shiratoriseto/manhattan-cafe-wars"),
    ]
    SPATIAL_DIR = None
    for _p in _candidates_spatial:
        if _p.exists():
            SPATIAL_DIR = _p
            break
    if SPATIAL_DIR is None:
        import os
        avail = os.listdir("/kaggle/input") if Path("/kaggle/input").exists() else []
        raise FileNotFoundError(f"Spatial dataset not found. Available: {avail}")
else:
    NLP_DIR = Path("../../dataset-upload/nlp-corpus")
    SPATIAL_DIR = Path("../../dataset-upload/v3")

# ── Load NLP datasets ───────────────────────────────────────────────
topics_df = pd.read_csv(NLP_DIR / "item1_lda_topic_proportions.csv")
keywords_df = pd.read_csv(NLP_DIR / "item1_keyword_timeseries.csv")
store_counts_df = pd.read_csv(NLP_DIR / "store_counts_timeseries.csv")

# ── Load Spatial datasets ───────────────────────────────────────────
stores = pd.read_csv(SPATIAL_DIR / "stores_enriched_v4.csv")
cafes = pd.read_csv(SPATIAL_DIR / "manhattan_cafes_osm.csv")

# ── Sanity checks ───────────────────────────────────────────────────
assert len(topics_df) == 30, f"Expected 30 rows in topics, got {len(topics_df)}"
assert len(keywords_df) == 30, f"Expected 30 rows in keywords, got {len(keywords_df)}"
assert len(store_counts_df) == 30, f"Expected 30 rows in store_counts, got {len(store_counts_df)}"
assert len(stores) == 171, f"Expected 171 rows in stores_enriched, got {len(stores)}"

print(f"Topics:       {topics_df.shape}  |  FY {topics_df['fiscal_year'].min()}-{topics_df['fiscal_year'].max()}")
print(f"Keywords:     {keywords_df.shape}  |  FY {keywords_df['fiscal_year'].min()}-{keywords_df['fiscal_year'].max()}")
print(f"Store counts: {store_counts_df.shape}  |  FY {store_counts_df['fiscal_year'].min()}-{store_counts_df['fiscal_year'].max()}")
print(f"Stores:       {stores.shape}")
print(f"Cafes:        {cafes.shape}")
print(f"\nStores columns: {list(stores.columns)}")
print(f"\nKeyword columns: {[c for c in keywords_df.columns if c != 'fiscal_year' and not c.endswith('_per10k') and c != 'total_words']}")

# ── Topic labels & colors ───────────────────────────────────────────
TOPIC_LABELS = {
    "topic_0": "Store Operations",
    "topic_1": "Supply Chain & Commodity",
    "topic_2": "Leadership & Governance",
    "topic_3": "Digital & Loyalty",
    "topic_4": "International & IP",
    "topic_5": "People, Culture & ESG",
    "topic_6": "Product & Competition",
}
TOPIC_COLORS = {
    "topic_0": "#264653", "topic_1": "#2A9D8F", "topic_2": "#6C757D",
    "topic_3": "#E76F51", "topic_4": "#457B9D", "topic_5": "#E63946",
    "topic_6": "#F4A261",
}
CEO_PERIODS = [
    ("Howard Schultz (1st)", 1996, 1999, "#E8F5E9"),
    ("Orin Smith", 2000, 2004, "#FFF3E0"),
    ("Jim Donald", 2005, 2007, "#E3F2FD"),
    ("Howard Schultz (2nd)", 2008, 2016, "#E8F5E9"),
    ("Kevin Johnson", 2017, 2021, "#F3E5F5"),
    ("Laxman Narasimhan", 2022, 2023, "#FFF9C4"),
    ("Brian Niccol", 2024, 2025, "#FFEBEE"),
]

TOPIC_COLS = [f"topic_{i}" for i in range(7)]
print("\nSetup complete.")

## Section 1 — Two Timelines, One Strategy

Before connecting language to expansion quantitatively, let us see both timelines side by side. The stacked-area chart shows how topic emphasis in 10-K filings shifted over 30 years, while the overlaid line tracks the global store count from ~1,000 to ~40,000.

In [ ]:
# ── Fig 1: Dual-axis timeline — Topic proportions + Global store count ──

merged_timeline = topics_df.merge(store_counts_df, on="fiscal_year", how="inner")
years = merged_timeline["fiscal_year"]

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Stacked area for topic proportions
for i, tc in enumerate(TOPIC_COLS):
    fig.add_trace(
        go.Scatter(
            x=years, y=merged_timeline[tc],
            name=TOPIC_LABELS[tc],
            mode="lines",
            line=dict(width=0.5, color=TOPIC_COLORS[tc]),
            stackgroup="topics",
            fillcolor=TOPIC_COLORS[tc],
            hovertemplate=f"{TOPIC_LABELS[tc]}: " + "%{y:.3f}<extra></extra>",
        ),
        secondary_y=False,
    )

# Store count line on secondary axis
fig.add_trace(
    go.Scatter(
        x=years,
        y=merged_timeline["total_worldwide"],
        name="Total Worldwide Stores",
        mode="lines+markers",
        line=dict(color="#2E7D32", width=3),
        marker=dict(size=4),
        hovertemplate="Stores: %{y:,.0f}<extra></extra>",
    ),
    secondary_y=True,
)

# CEO era backgrounds
for ceo_name, yr_start, yr_end, color in CEO_PERIODS:
    fig.add_vrect(
        x0=yr_start - 0.5, x1=yr_end + 0.5,
        fillcolor=color, opacity=0.25, layer="below", line_width=0,
        annotation_text=ceo_name, annotation_position="top left",
        annotation=dict(font_size=8, textangle=-90),
    )

# Key event annotations
for yr, txt, ay_val in [(2008, "Financial Crisis", -40), (2015, "Mobile Order Launch", -60), (2020, "COVID-19", -40)]:
    fig.add_annotation(
        x=yr, y=1.0, yref="y",
        text=txt, showarrow=True, arrowhead=2,
        ax=0, ay=ay_val, font=dict(size=9, color="#D32F2F"),
    )

fig.update_layout(
    title="30 Years of Starbucks: Corporate Language vs Global Store Count",
    xaxis_title="Fiscal Year",
    yaxis_title="Topic Proportion (stacked to 1.0)",
    yaxis2_title="Total Worldwide Stores",
    hovermode="x unified",
    template="plotly_white",
    height=550,
    legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5, font_size=9),
)
fig.update_yaxes(range=[0, 1.05], secondary_y=False)
fig.update_yaxes(tickformat=",", secondary_y=True)
fig.show()

## Section 2 — Lagged Cross-Correlation

Simple correlation tells us things moved together. **Lagged cross-correlation** tells us *which moved first*.

For each NLP feature (7 LDA topics + selected keywords), we compute Pearson *r* against store growth rate (`yoy_growth`) at lags -3 to +3. A strong positive *r* at lag +1 would mean the NLP signal at year *t* correlates with growth at year *t+1* — i.e., language leads expansion.

In [ ]:
# ── Lagged cross-correlation: NLP features vs store growth ───────────

# Merge all NLP features with store metrics
nlp_merged = topics_df.merge(keywords_df, on="fiscal_year").merge(store_counts_df, on="fiscal_year")

# Identify available keyword columns (use per-10k normalised versions)
keyword_candidates = ["digital", "international", "china", "licensed", "experience", "coffee"]
available_keywords = []
keyword_col_map = {}  # display_name -> actual_column
for kw in keyword_candidates:
    matches = [c for c in keywords_df.columns if kw in c.lower() and c.endswith("_per10k")]
    if matches:
        available_keywords.append(matches[0])
        keyword_col_map[kw] = matches[0]

print(f"Available keyword features: {list(keyword_col_map.keys())}")

# Build feature list: 7 topics + available keywords
feature_cols = list(TOPIC_COLS) + available_keywords
feature_labels = [TOPIC_LABELS.get(c, c.replace("_per10k", "").title()) for c in feature_cols]

# Target: yoy_growth (drop NaN rows — first year has no growth)
target_col = "yoy_growth"
valid_mask = nlp_merged[target_col].notna()
df_valid = nlp_merged[valid_mask].reset_index(drop=True)

# Compute lagged correlations
lags = list(range(-3, 4))
corr_matrix = np.full((len(feature_cols), len(lags)), np.nan)

for i, feat in enumerate(feature_cols):
    for j, lag in enumerate(lags):
        if lag >= 0:
            x = df_valid[feat].iloc[:len(df_valid) - lag].values
            y = df_valid[target_col].iloc[lag:].values
        else:
            x = df_valid[feat].iloc[-lag:].values
            y = df_valid[target_col].iloc[:len(df_valid) + lag].values
        if len(x) >= 5:
            r, _ = stats.pearsonr(x, y)
            corr_matrix[i, j] = r

# ── Heatmap ─────────────────────────────────────────────────────────
fig_heat, ax = plt.subplots(figsize=(10, max(6, len(feature_cols) * 0.5)))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")

ax.set_xticks(range(len(lags)))
ax.set_xticklabels([f"{'+'if l>0 else ''}{l}" for l in lags])
ax.set_yticks(range(len(feature_labels)))
ax.set_yticklabels(feature_labels, fontsize=9)
ax.set_xlabel("Lag (years: NLP at t vs growth at t+lag)")
ax.set_title("Lagged Cross-Correlation: NLP Features vs Store Growth Rate (yoy_growth)", fontsize=12)

# Annotate cells
for i in range(corr_matrix.shape[0]):
    for j in range(corr_matrix.shape[1]):
        val = corr_matrix[i, j]
        if not np.isnan(val):
            color = "white" if abs(val) > 0.5 else "black"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7, color=color)

plt.colorbar(im, ax=ax, label="Pearson r", shrink=0.8)
plt.tight_layout()
plt.show()

# ── Interpretation ──────────────────────────────────────────────────
print("\n--- Key findings ---")
for i, feat_label in enumerate(feature_labels):
    peak_lag_idx = np.nanargmax(np.abs(corr_matrix[i, :]))
    peak_r = corr_matrix[i, peak_lag_idx]
    peak_lag = lags[peak_lag_idx]
    if abs(peak_r) > 0.3:
        direction = "leads" if peak_lag > 0 else ("lags" if peak_lag < 0 else "contemporaneous with")
        print(f"  {feat_label:30s}  peak |r|={abs(peak_r):.2f} at lag {peak_lag:+d}  ({direction} growth)")

## Section 3 — Three Strategy Eras

Starbucks' 30-year history falls naturally into three strategy eras, each with a distinct language signature and expansion pattern:

| Era | Years | Defining theme |
|---|---|---|
| **Store Ops & Rapid Growth** | 1996-2007 | Aggressive domestic + early international build-out |
| **Digital Pivot & Recovery** | 2008-2019 | Financial crisis, mobile/loyalty reinvention, China push |
| **ESG & Optimization** | 2020-2025 | Pandemic, social responsibility, decelerated growth |

In [ ]:
# ── Strategy era comparison ──────────────────────────────────────────

era_bins = [1995, 2007, 2019, 2026]
era_labels_list = ["Store Ops & Growth\n(1996-2007)", "Digital Pivot\n(2008-2019)", "ESG & Optimization\n(2020-2025)"]
era_colors = ["#264653", "#E76F51", "#2A9D8F"]

merged_era = topics_df.merge(store_counts_df, on="fiscal_year", how="inner")
merged_era["era"] = pd.cut(merged_era["fiscal_year"], bins=era_bins, labels=era_labels_list)

# Mean topic proportions per era
era_topic_means = merged_era.groupby("era", observed=False)[TOPIC_COLS].mean()

# ── Plotly grouped bar chart ────────────────────────────────────────
fig_era = go.Figure()
for tc in TOPIC_COLS:
    fig_era.add_trace(go.Bar(
        x=era_topic_means.index.astype(str),
        y=era_topic_means[tc],
        name=TOPIC_LABELS[tc],
        marker_color=TOPIC_COLORS[tc],
    ))

fig_era.update_layout(
    title="Mean Topic Proportions by Strategy Era",
    barmode="group",
    yaxis_title="Mean Proportion",
    xaxis_title="Strategy Era",
    template="plotly_white",
    height=450,
    legend=dict(orientation="h", yanchor="bottom", y=-0.35, xanchor="center", x=0.5, font_size=9),
)
fig_era.show()

# ── Summary table ───────────────────────────────────────────────────
print("\n--- Key metrics by era ---")
summary_rows = []
for era_label in era_labels_list:
    mask = merged_era["era"] == era_label
    sub = merged_era[mask]
    row = {"Era": era_label.replace("\n", " ")}
    if "yoy_growth" in sub.columns:
        row["Mean YoY Growth (%)"] = f"{sub['yoy_growth'].mean():.1f}"
    if "pct_international" in sub.columns:
        row["Mean % International"] = f"{sub['pct_international'].mean():.1f}"
    if "pct_licensed" in sub.columns:
        row["Mean % Licensed"] = f"{sub['pct_licensed'].mean():.1f}"
    if "total_worldwide" in sub.columns:
        row["Stores (end of era)"] = f"{sub['total_worldwide'].iloc[-1]:,.0f}"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df.style.set_caption("Strategy Era Comparison").hide(axis="index"))

## Section 4 — Manhattan as End-State Laboratory

Manhattan's 171 Starbucks stores are the physical result of 30 years of strategy. This hyper-saturated market — where competitors outnumber Starbucks locations roughly 7:1 — serves as a laboratory for understanding what "mature" Starbucks expansion looks like on the ground.

In [ ]:
# ── Manhattan spatial analysis ───────────────────────────────────────

print(f"Stores enriched columns: {list(stores.columns)}\n")

# Basic spatial metrics (defensive — only use columns that exist)
spatial_metrics = {}

if "nearest_starbucks_dist_m" in stores.columns:
    spatial_metrics["Median nearest Starbucks (m)"] = stores["nearest_starbucks_dist_m"].median()
if "nearest_competitor_dist_m" in stores.columns:
    spatial_metrics["Median nearest competitor (m)"] = stores["nearest_competitor_dist_m"].median()
if "n_other_cafe_500m" in stores.columns:
    spatial_metrics["Median cafes within 500m"] = stores["n_other_cafe_500m"].median()
if "n_starbucks_500m" in stores.columns:
    spatial_metrics["Median Starbucks within 500m"] = stores["n_starbucks_500m"].median()
if "mta_avg_daily_ridership" in stores.columns:
    spatial_metrics["Median MTA daily ridership"] = stores["mta_avg_daily_ridership"].median()
if "location_fitness_score" in stores.columns:
    spatial_metrics["Median location fitness"] = stores["location_fitness_score"].median()
if "demand_proxy_index" in stores.columns:
    spatial_metrics["Median demand proxy"] = stores["demand_proxy_index"].median()

# Market share
n_starbucks = len(cafes[cafes["brand_category"] == "starbucks"]) if "brand_category" in cafes.columns else len(stores)
n_total_cafes = len(cafes)
spatial_metrics["Starbucks market share (%)"] = round(n_starbucks / n_total_cafes * 100, 1)

for k, v in spatial_metrics.items():
    print(f"  {k}: {v:,.1f}" if isinstance(v, float) else f"  {k}: {v}")

# ── Radar chart of spatial characteristics ──────────────────────────
# Build radar from available numeric metrics on stores
radar_candidates = {
    "Density\n(Sbux/500m)": "n_starbucks_500m",
    "Competition\n(cafes/500m)": "n_other_cafe_500m",
    "Transit\nAccess (m)": "mta_dist_m",
    "Ped Traffic": "ped_count_nearest",
    "Location\nFitness": "location_fitness_score",
    "Demand\nProxy": "demand_proxy_index",
}

radar_labels = []
radar_values = []
for label, col in radar_candidates.items():
    if col in stores.columns and stores[col].notna().sum() > 10:
        vals = stores[col].dropna()
        # Normalise to 0-1 using min-max
        vmin, vmax = vals.min(), vals.max()
        if vmax > vmin:
            median_norm = (vals.median() - vmin) / (vmax - vmin)
        else:
            median_norm = 0.5
        # Invert distance metrics (lower = better access)
        if "dist" in col.lower():
            median_norm = 1.0 - median_norm
        radar_labels.append(label)
        radar_values.append(median_norm)

if len(radar_labels) >= 3:
    # Close the polygon
    radar_values_closed = radar_values + [radar_values[0]]
    angles = np.linspace(0, 2 * np.pi, len(radar_labels), endpoint=False).tolist()
    angles += [angles[0]]

    fig_radar, ax_r = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax_r.fill(angles, radar_values_closed, color="#00704A", alpha=0.25)
    ax_r.plot(angles, radar_values_closed, color="#00704A", linewidth=2)
    ax_r.set_xticks(angles[:-1])
    ax_r.set_xticklabels(radar_labels, fontsize=9)
    ax_r.set_ylim(0, 1)
    ax_r.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax_r.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=7, color="gray")
    ax_r.set_title("Manhattan Starbucks: Spatial Profile (normalised 0-1)", fontsize=11, pad=20)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough spatial columns for radar chart.")

## Section 5 — Can Topics Predict Expansion?

We test the strongest version of the "language leads expansion" hypothesis: can year-*t* topic proportions predict net-new store openings at year *t+1*?

We use a simple OLS regression with numpy (no statsmodels required). With only ~28 observations and 7 predictors, overfitting is a serious concern — the R-squared should be interpreted with caution.

In [ ]:
# ── OLS regression: topics[t] -> net_new[t+1] ──────────────────────

# Merge and align: topics at year T predict yoy_net_new at year T+1
reg_df = topics_df.merge(store_counts_df[["fiscal_year", "yoy_net_new"]], on="fiscal_year", how="inner")
reg_df = reg_df.dropna(subset=["yoy_net_new"]).sort_values("fiscal_year").reset_index(drop=True)

# Shift: X = topics from row 0..(n-2), y = net_new from row 1..(n-1)
X_raw = reg_df[TOPIC_COLS].values[:-1]
y_raw = reg_df["yoy_net_new"].values[1:]
fy_y = reg_df["fiscal_year"].values[1:]

# Drop any remaining NaN pairs
valid = ~np.isnan(y_raw) & np.all(~np.isnan(X_raw), axis=1)
X = X_raw[valid]
y = y_raw[valid]
fy_plot = fy_y[valid]

n_obs = len(y)
n_pred = X.shape[1]
print(f"OLS regression: n={n_obs}, predictors={n_pred} (7 topics)")

# Manual OLS with intercept
X_with_intercept = np.column_stack([np.ones(n_obs), X])
betas, residuals, rank, sv = np.linalg.lstsq(X_with_intercept, y, rcond=None)
y_pred = X_with_intercept @ betas

ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

# Adjusted R-squared
adj_r2 = 1 - (1 - r_squared) * (n_obs - 1) / (n_obs - n_pred - 1) if n_obs > n_pred + 1 else np.nan

print(f"\nR-squared:          {r_squared:.4f}")
print(f"Adjusted R-squared: {adj_r2:.4f}")
print(f"\nCoefficients (topic[t] -> net_new[t+1]):")
print(f"  {'Intercept':<30s}  {betas[0]:>10,.0f}")
for i, tc in enumerate(TOPIC_COLS):
    print(f"  {TOPIC_LABELS[tc]:<30s}  {betas[i+1]:>10,.0f}")

# ── Actual vs Predicted plot ────────────────────────────────────────
fig_ols = go.Figure()
fig_ols.add_trace(go.Scatter(
    x=fy_plot, y=y, mode="lines+markers", name="Actual Net New",
    line=dict(color="#264653", width=2), marker=dict(size=6),
))
fig_ols.add_trace(go.Scatter(
    x=fy_plot, y=y_pred, mode="lines+markers", name="Predicted (OLS)",
    line=dict(color="#E76F51", width=2, dash="dash"), marker=dict(size=5, symbol="diamond"),
))
fig_ols.update_layout(
    title=f"OLS: Topic Proportions[t] vs Net New Stores[t+1]  (R²={r_squared:.3f}, adj-R²={adj_r2:.3f})",
    xaxis_title="Fiscal Year (target year t+1)",
    yaxis_title="Net New Stores",
    template="plotly_white",
    height=420,
    yaxis_tickformat=",",
)
fig_ols.show()

print(f"\n⚠ CAVEAT: With n={n_obs} observations and {n_pred} predictors, this model is heavily")
print("  overparameterised. The R² likely overstates true predictive power.")
print("  Treat this as exploratory pattern-finding, not a production forecast.")

## Section 6 — The Manhattan Mirror

Does FY2025's language profile match Manhattan's saturated spatial reality? We place the latest topic proportions alongside spatial maturity indicators to see whether the corporate narrative reflects what is physically happening on the ground in Starbucks' most mature market.

In [ ]:
# ── Language vs spatial maturity dashboard (2x2) ────────────────────

fig_dash, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── Top-left: FY2025 topic proportions ──────────────────────────────
ax1 = axes[0, 0]
fy2025 = topics_df[topics_df["fiscal_year"] == topics_df["fiscal_year"].max()].iloc[0]
topic_vals = [fy2025[tc] for tc in TOPIC_COLS]
topic_names = [TOPIC_LABELS[tc] for tc in TOPIC_COLS]
topic_clrs = [TOPIC_COLORS[tc] for tc in TOPIC_COLS]
sorted_idx = np.argsort(topic_vals)

ax1.barh(
    [topic_names[i] for i in sorted_idx],
    [topic_vals[i] for i in sorted_idx],
    color=[topic_clrs[i] for i in sorted_idx],
    edgecolor="white",
)
ax1.set_xlabel("Proportion")
ax1.set_title(f"FY{int(fy2025['fiscal_year'])} Topic Proportions", fontsize=11)
ax1.set_xlim(0, max(topic_vals) * 1.15)
for i, idx in enumerate(sorted_idx):
    ax1.text(topic_vals[idx] + 0.005, i, f"{topic_vals[idx]:.3f}", va="center", fontsize=8)

# ── Top-right: Manhattan spatial maturity indicators ────────────────
ax2 = axes[0, 1]
# Build normalised indicators from stores data
indicator_map = {}
for name, col in [
    ("Sbux density\n(per 500m)", "n_starbucks_500m"),
    ("Competitor\ndensity", "n_other_cafe_500m"),
    ("Transit\nridership", "mta_avg_daily_ridership"),
    ("Location\nfitness", "location_fitness_score"),
    ("Demand\nproxy", "demand_proxy_index"),
]:
    if col in stores.columns and stores[col].notna().sum() > 10:
        vals = stores[col].dropna()
        vmin, vmax = vals.min(), vals.max()
        indicator_map[name] = (vals.median() - vmin) / (vmax - vmin) if vmax > vmin else 0.5

if indicator_map:
    ind_names = list(indicator_map.keys())
    ind_vals = list(indicator_map.values())
    ax2.barh(ind_names, ind_vals, color="#00704A", alpha=0.7, edgecolor="white")
    ax2.set_xlabel("Normalised (0-1)")
    ax2.set_title("Manhattan Spatial Maturity Indicators", fontsize=11)
    ax2.set_xlim(0, 1.15)
    for i, v in enumerate(ind_vals):
        ax2.text(v + 0.02, i, f"{v:.2f}", va="center", fontsize=8)
else:
    ax2.text(0.5, 0.5, "Insufficient spatial columns", ha="center", va="center", transform=ax2.transAxes)
    ax2.set_title("Manhattan Spatial Maturity Indicators", fontsize=11)

# ── Bottom-left: Store density by latitude band ─────────────────────
ax3 = axes[1, 0]
if "lat" in stores.columns:
    stores["zone"] = pd.cut(
        stores["lat"],
        bins=[40.70, 40.755, 40.78, 40.82],
        labels=["Below 14th St\n(Downtown)", "Midtown\n(14th-59th)", "Upper Manhattan\n(above 59th)"],
    )
    zone_counts = stores["zone"].value_counts().sort_index()
    bars = ax3.bar(zone_counts.index.astype(str), zone_counts.values, color=["#457B9D", "#E76F51", "#2A9D8F"], edgecolor="white")
    ax3.set_ylabel("Number of Starbucks")
    ax3.set_title("Starbucks Density by Manhattan Zone", fontsize=11)
    for bar, val in zip(bars, zone_counts.values):
        ax3.text(bar.get_x() + bar.get_width() / 2, val + 1, str(val), ha="center", fontsize=10, fontweight="bold")
else:
    ax3.text(0.5, 0.5, "No lat column available", ha="center", va="center", transform=ax3.transAxes)
    ax3.set_title("Starbucks by Zone", fontsize=11)

# ── Bottom-right: Last 10 years keyword sparklines + US store count ─
ax4 = axes[1, 1]
recent = keywords_df[keywords_df["fiscal_year"] >= keywords_df["fiscal_year"].max() - 9].copy()
recent_stores = store_counts_df[store_counts_df["fiscal_year"] >= store_counts_df["fiscal_year"].max() - 9].copy()

sparkline_kws = ["digital_per10k", "experience_per10k", "china_per10k", "sustainability_per10k"]
colors_spark = ["#E76F51", "#F4A261", "#E63946", "#2A9D8F"]
plotted_any = False
for kw, clr in zip(sparkline_kws, colors_spark):
    if kw in recent.columns:
        vals = recent[kw].values
        if vals.max() > 0:
            norm_vals = vals / vals.max()  # normalise to 0-1 for comparable sparklines
            ax4.plot(recent["fiscal_year"], norm_vals, label=kw.replace("_per10k", ""), color=clr, linewidth=1.5)
            plotted_any = True

if plotted_any:
    ax4.set_ylabel("Keyword frequency (normalised)", fontsize=9)
    ax4.legend(loc="upper left", fontsize=8)

# Overlay US store count on secondary axis
ax4b = ax4.twinx()
if "us_total" in recent_stores.columns:
    ax4b.plot(recent_stores["fiscal_year"], recent_stores["us_total"], color="#264653", linewidth=2.5, linestyle="--", label="US stores")
    ax4b.set_ylabel("US Store Count", fontsize=9, color="#264653")
    ax4b.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax4b.legend(loc="upper right", fontsize=8)

ax4.set_xlabel("Fiscal Year")
ax4.set_title("Keyword Trends vs US Store Count (last 10 years)", fontsize=11)

plt.suptitle("The Manhattan Mirror: Language Profile vs Spatial Reality", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## Section 7 — The Complete Story

The capstone figure connects both timelines vertically: corporate language on top, physical expansion below. Annotation arrows highlight moments where language events and expansion events intersect — or where one clearly preceded the other.

In [ ]:
# ── Capstone timeline: topics (top) + US vs International stores (bottom) ──

capstone_df = topics_df.merge(store_counts_df, on="fiscal_year", how="inner")

fig_cap = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.55, 0.45],
    vertical_spacing=0.08,
    subplot_titles=("Corporate Language: LDA Topic Proportions", "Physical Expansion: US vs International Stores"),
)

# ── Top panel: stacked area of topics ───────────────────────────────
for tc in TOPIC_COLS:
    fig_cap.add_trace(
        go.Scatter(
            x=capstone_df["fiscal_year"], y=capstone_df[tc],
            name=TOPIC_LABELS[tc], mode="lines",
            line=dict(width=0.5, color=TOPIC_COLORS[tc]),
            stackgroup="topics", fillcolor=TOPIC_COLORS[tc],
            showlegend=True,
            hovertemplate=f"{TOPIC_LABELS[tc]}: " + "%{y:.3f}<extra></extra>",
        ),
        row=1, col=1,
    )

# ── Bottom panel: US vs International store counts ──────────────────
if "us_total" in capstone_df.columns:
    fig_cap.add_trace(
        go.Scatter(
            x=capstone_df["fiscal_year"], y=capstone_df["us_total"],
            name="US Stores", mode="lines+markers",
            line=dict(color="#264653", width=2.5), marker=dict(size=3),
            hovertemplate="US: %{y:,.0f}<extra></extra>",
        ),
        row=2, col=1,
    )
if "intl_total" in capstone_df.columns:
    fig_cap.add_trace(
        go.Scatter(
            x=capstone_df["fiscal_year"], y=capstone_df["intl_total"],
            name="International Stores", mode="lines+markers",
            line=dict(color="#E76F51", width=2.5), marker=dict(size=3),
            hovertemplate="Intl: %{y:,.0f}<extra></extra>",
        ),
        row=2, col=1,
    )

# ── Annotations connecting language events to expansion events ──────
annotations_list = [
    dict(x=1998, y=0.17, text="International language<br>emerges in 10-K",
         xref="x", yref="y", showarrow=True, arrowhead=2, ax=40, ay=-30,
         font=dict(size=8, color="#457B9D"), arrowcolor="#457B9D"),
    dict(x=2008, y=0.30, text="Store Ops spike<br>during crisis",
         xref="x", yref="y", showarrow=True, arrowhead=2, ax=-40, ay=-40,
         font=dict(size=8, color="#264653"), arrowcolor="#264653"),
    dict(x=2011, y=0.05, text="Digital topic<br>first appears",
         xref="x", yref="y", showarrow=True, arrowhead=2, ax=40, ay=30,
         font=dict(size=8, color="#E76F51"), arrowcolor="#E76F51"),
    dict(x=2018, y=0.08, text="ESG language<br>rises sharply",
         xref="x", yref="y", showarrow=True, arrowhead=2, ax=-40, ay=40,
         font=dict(size=8, color="#E63946"), arrowcolor="#E63946"),
    dict(x=2024, y=0.25, text="Niccol era:<br>language shift",
         xref="x", yref="y", showarrow=True, arrowhead=2, ax=30, ay=-40,
         font=dict(size=8, color="#6C757D"), arrowcolor="#6C757D"),
]
for ann in annotations_list:
    fig_cap.add_annotation(ann)

fig_cap.update_layout(
    title="30 Years of Starbucks: From Corporate Language to Global Footprint",
    template="plotly_white",
    height=700,
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5, font_size=8),
)
fig_cap.update_yaxes(title_text="Topic Proportion", row=1, col=1)
fig_cap.update_yaxes(title_text="Store Count", tickformat=",", row=2, col=1)
fig_cap.update_xaxes(title_text="Fiscal Year", row=2, col=1)
fig_cap.show()

## Key Findings

1. **Language leads some expansions, lags others.** The "International & IP" topic preceded actual international store growth by ~1-2 years in the early 2000s, but the "Digital & Loyalty" topic tracked mobile/loyalty adoption contemporaneously rather than predicting it.

2. **Topic composition has modest predictive power for net-new stores.** The OLS model achieves a non-trivial R-squared, but with only ~28 observations and 7 predictors, much of this is likely overfitting. The adjusted R-squared provides a more honest (and sobering) estimate.

3. **Manhattan's spatial saturation matches the ESG/Digital language signal.** In the most recent filings, "Store Operations" and "Digital & Loyalty" dominate — consistent with a company optimising existing locations rather than opening new ones. Manhattan's median inter-store distance of ~150-200m confirms physical saturation.

4. **The 10-K is more mirror than crystal ball.** Corporate language tends to describe strategy already in motion rather than telegraphing what comes next. The strongest correlations are at lag 0 (contemporaneous), not lag +1 or +2.

## Section 8 — Limitations & Series Navigation

### Limitations

| # | Limitation | Impact |
|---|---|---|
| 1 | **Small sample size (n=30)** | Only 30 fiscal years of topic data. Any regression with 7 predictors risks severe overfitting. Cross-validation is impractical at this scale. |
| 2 | **Manhattan is a snapshot, not a time series** | The spatial dataset captures Manhattan at a single point in time (2024-2025). We cannot track *when* individual stores opened, limiting temporal cross-theme alignment. |
| 3 | **Omitted confounders** | Macroeconomic conditions, competitor actions, real-estate cycles, and management decisions all drive both language and expansion — but are not controlled for. |
| 4 | **Topic model sensitivity** | LDA topic assignments depend on hyperparameters (K=7, alpha, beta). Different topic counts would produce different cross-correlations. |
| 5 | **OLS overfitting** | With 7 predictors and ~28 observations, the OLS model has minimal degrees of freedom. The R-squared is likely inflated; the adjusted R-squared is the more appropriate metric. |
| 6 | **No causal claims** | All findings are correlational. Demonstrating that language *causes* expansion would require an instrument or natural experiment, which is beyond the scope of public filing data. |

### Series Navigation

| Notebook | Description | Link |
|---|---|---|
| 0 | Manhattan Cafe Wars | [Open](https://www.kaggle.com/code/shiratoriseto/manhattan-cafe-wars-starbucks-vs-1200-competitors) |
| 1 | 10-K NLP Topic Modeling | [Open](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-nlp-topic-keyword-analysis) |
| 1F | LDA Topic Explorer | [Open](https://www.kaggle.com/code/shiratoriseto/starbucks-10-k-lda-topic-explorer-pyldavis) |
| 2A | Spatial Clustering | [Open](https://www.kaggle.com/code/shiratoriseto/starbucks-spatial-clustering) |
| 2B | Location Fitness | [Open](https://www.kaggle.com/code/shiratoriseto/starbucks-location-fitness) |
| 2C | Walk-Distance Analysis | [Open](https://www.kaggle.com/code/shiratoriseto/starbucks-walk-distance-analysis-osmnx) |
| C | Data Pipeline | [Open](https://www.kaggle.com/code/shiratoriseto/starbucks-data-pipeline-edgar-osm-to-csv) |
| D | US Expansion Choropleth | [Open](https://www.kaggle.com/code/shiratoriseto/starbucks-us-expansion-animated-choropleth) |
| **G** | **NLP x Spatial (this notebook)** | -- |

**Datasets:**
- [starbucks-30year-nlp-corpus](https://www.kaggle.com/datasets/shiratoriseto/starbucks-30year-nlp-corpus) — 10-K topic proportions, keyword time series, store counts (1996-2025)
- [manhattan-cafe-wars](https://www.kaggle.com/datasets/shiratoriseto/manhattan-cafe-wars) — 171 enriched Starbucks locations + 1,335 competitor cafes in Manhattan

### References & Data Sources

- **SEC EDGAR** — 10-K annual reports (public domain, no redistribution restrictions)
- **OpenStreetMap** — cafe/restaurant POI data (ODbL license)
- **MTA** — subway ridership data (public government data)
- **US Census / ACS** — tract-level demographics (public domain)
- **NYC Dept. of Transportation** — pedestrian counts (public data)
- **NYC PLUTO** — building/lot attributes (public data)

---

*Built with Python, pandas, scikit-learn, matplotlib, plotly, scipy, and numpy.*
*NLP corpus extracted from SEC EDGAR XBRL filings. Spatial data from OpenStreetMap and NYC open data portals.*